## Notebook 10 — Feature Correlation Analysis
**Project:** Machine Learning for High Performance Optical Sorting
**Author:** Mohamed Tawfeek
**Description:** Pairwise Pearson correlation analysis of the 122-dimensional HSV+LBP feature space, including zero-variance feature identification and group-level block summaries


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
from sklearn.model_selection import train_test_split

In [2]:
FEATURES_PATH = os.path.join("..", "results", "features", "features.npy")
LABELS_PATH   = os.path.join("..", "results", "features", "labels.npy")
FIGURES_DIR   = os.path.join("..", "results", "figures")

In [3]:
X = np.load(FEATURES_PATH)
y = np.load(LABELS_PATH)
print(f"Features: {X.shape}  Labels: {y.shape}")

Features: (2527, 122)  Labels: (2527,)


In [4]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.176, random_state=42, stratify=y_train_val
)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

Train: (1769, 122)  Val: (378, 122)  Test: (380, 122)


In [5]:
group_boundaries = [0, 32, 64, 96, 122]
group_names      = ["H (Hue)", "S (Sat)", "V (Val)", "LBP"]
group_colors     = ["#2196F3", "#4CAF50", "#FF9800", "#9C27B0"]

In [6]:
# Identifies features with zero standard deviation across all training samples — H histogram bins for hues absent in the TrashNet dataset

stds = X_train.std(axis=0)
zero_var = np.where(stds == 0)[0]
print(f"\nZero-variance features (constant across all training images):")
print(f"  Indices: {zero_var.tolist()}")
print(f"  Count:   {len(zero_var)}")
print(f"  These are H bins with no representation in any TrashNet class.")


Zero-variance features (constant across all training images):
  Indices: [23, 24, 25, 26, 27, 28, 29, 30, 31]
  Count:   9
  These are H bins with no representation in any TrashNet class.


In [7]:
# Computes the 122×122 Pearson correlation matrix; zero-variance features produce NaN entries, which are tracked separately

with np.errstate(invalid="ignore", divide="ignore"):
    corr = np.corrcoef(X_train.T)
 
nan_count = np.isnan(corr).sum()
print(f"\nNaN cells in correlation matrix: {nan_count} "
      f"(expected: {2 * len(zero_var) * 122 - len(zero_var)**2})")


NaN cells in correlation matrix: 2115 (expected: 2115)


In [8]:
# Replaces NaN correlation entries with 0.0 so they render as a neutral colour in the heatmap rather than being masked

corr_display = np.where(np.isnan(corr), 0.0, corr)

In [9]:
# Plots the full 122×122 correlation heatmap with group boundary lines and zero-variance feature bands annotated

fig, ax = plt.subplots(figsize=(11, 9))
 
cmap = plt.cm.RdBu_r.copy()
cmap.set_bad(color="#cccccc")   # NaN cells → grey (not needed after fill but kept)
 
im = ax.imshow(corr_display, cmap=cmap, vmin=-1, vmax=1, aspect="auto")
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Pearson Correlation (grey = zero-variance feature)", fontsize=8)
 
for idx in zero_var:
    ax.axvline(idx, color="#cccccc", linewidth=0.8, alpha=0.9)
    ax.axhline(idx, color="#cccccc", linewidth=0.8, alpha=0.9)
 
for b in group_boundaries[1:-1]:
    ax.axvline(b - 0.5, color="black", linewidth=1.5, linestyle="--", alpha=0.7)
    ax.axhline(b - 0.5, color="black", linewidth=1.5, linestyle="--", alpha=0.7)
 
tick_positions = [(group_boundaries[i] + group_boundaries[i+1]) / 2
                  for i in range(len(group_names))]
ax.set_xticks(tick_positions)
ax.set_xticklabels(group_names, fontsize=9)
ax.set_yticks(tick_positions)
ax.set_yticklabels(group_names, fontsize=9)
for tick, col in zip(ax.get_xticklabels(), group_colors):
    tick.set_color(col); tick.set_fontweight("bold")
for tick, col in zip(ax.get_yticklabels(), group_colors):
    tick.set_color(col); tick.set_fontweight("bold")
 
ax.set_title(
    "Feature Correlation Matrix — HSV + LBP (122 features, training set)\n"
    "Grey bands = zero-variance H bins (cyan–violet hues absent in TrashNet)",
    fontsize=10, pad=12
)
 
plt.tight_layout()
out1 = os.path.join(FIGURES_DIR, "feature_correlation_heatmap.png")
plt.savefig(out1, dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved: {out1}")


Saved: ..\results\figures\feature_correlation_heatmap.png


In [10]:
# Aggregates the full correlation matrix into a 4×4 block summary by computing mean absolute Pearson r within and between each feature group

n_groups = len(group_names)
block_corr_abs = np.zeros((n_groups, n_groups))
 
for i in range(n_groups):
    for j in range(n_groups):
        lo_i, hi_i = group_boundaries[i], group_boundaries[i+1]
        lo_j, hi_j = group_boundaries[j], group_boundaries[j+1]
        block = corr[lo_i:hi_i, lo_j:hi_j]
 
        if i == j:
            mask = ~np.eye(block.shape[0], dtype=bool)
            vals = block[mask]
        else:
            vals = block.flatten()
 
        valid = vals[~np.isnan(vals)]
        block_corr_abs[i, j] = np.abs(valid).mean() if len(valid) > 0 else 0.0
 
print("\nBlock-level mean absolute correlation (NaN-excluded, valid features only):")
print(f"{'':16}", end="")
for n in group_names:
    print(f"{n:>12}", end="")
print()
for i, ni in enumerate(group_names):
    print(f"{ni:16}", end="")
    for j in range(n_groups):
        print(f"{block_corr_abs[i,j]:>12.3f}", end="")
    print()


Block-level mean absolute correlation (NaN-excluded, valid features only):
                     H (Hue)     S (Sat)     V (Val)         LBP
H (Hue)                0.142       0.069       0.060       0.066
S (Sat)                0.069       0.184       0.091       0.133
V (Val)                0.060       0.091       0.213       0.133
LBP                    0.066       0.133       0.133       0.531


In [11]:
# Plots the compact 4×4 group-level mean absolute correlation heatmap

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(block_corr_abs, cmap="Oranges", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Mean |Pearson r|")
ax.set_xticks(range(n_groups))
ax.set_xticklabels(group_names, fontsize=10)
ax.set_yticks(range(n_groups))
ax.set_yticklabels(group_names, fontsize=10)
for ti, col in zip(ax.get_xticklabels(), group_colors):
    ti.set_color(col); ti.set_fontweight("bold")
for ti, col in zip(ax.get_yticklabels(), group_colors):
    ti.set_color(col); ti.set_fontweight("bold")
for i in range(n_groups):
    for j in range(n_groups):
        ax.text(j, i, f"{block_corr_abs[i,j]:.3f}",
                ha="center", va="center", fontsize=10,
                color="white" if block_corr_abs[i,j] > 0.5 else "black")
ax.set_title("Group-Level Mean Absolute Correlation\n(training set, zero-variance features excluded)",
             fontsize=10)
plt.tight_layout()
out2 = os.path.join(FIGURES_DIR, "feature_correlation_block_summary.png")
plt.savefig(out2, dpi=150, bbox_inches="tight")
plt.close()
print(f"\nSaved: {out2}")


Saved: ..\results\figures\feature_correlation_block_summary.png


In [12]:
print("\n" + "=" * 30)
print("KEY FINDINGS")
print("=" * 30)
print("\nWithin-group mean absolute correlations (valid features only):")
for i, name in enumerate(group_names):
    lo, hi = group_boundaries[i], group_boundaries[i+1]
    block  = corr[lo:hi, lo:hi]
    mask   = ~np.eye(block.shape[0], dtype=bool)
    vals   = block[mask].flatten()
    valid  = vals[~np.isnan(vals)]
    print(f"  {name:<16}: mean |r| = {np.abs(valid).mean():.3f}  "
          f"  median |r| = {np.median(np.abs(valid)):.3f}")
 
print("\nCross-group mean absolute correlations:")
for i in range(n_groups):
    for j in range(i+1, n_groups):
        print(f"  {group_names[i]:<14} vs {group_names[j]:<14}: "
              f"mean |r| = {block_corr_abs[i,j]:.3f}")


KEY FINDINGS

Within-group mean absolute correlations (valid features only):
  H (Hue)         : mean |r| = 0.142    median |r| = 0.104
  S (Sat)         : mean |r| = 0.184    median |r| = 0.092
  V (Val)         : mean |r| = 0.213    median |r| = 0.151
  LBP             : mean |r| = 0.531    median |r| = 0.552

Cross-group mean absolute correlations:
  H (Hue)        vs S (Sat)       : mean |r| = 0.069
  H (Hue)        vs V (Val)       : mean |r| = 0.060
  H (Hue)        vs LBP           : mean |r| = 0.066
  S (Sat)        vs V (Val)       : mean |r| = 0.091
  S (Sat)        vs LBP           : mean |r| = 0.133
  V (Val)        vs LBP           : mean |r| = 0.133
